# Multi-Agent Collaboration Patterns
Companion notebook for the **Multi-Agent Collaboration Patterns** section of Chapter 11.

Split work when the task has clear roles, different skills, or separate risk levels. The simple pattern: **planner, worker, reviewer**. Pass a structured state object between them -- never just the raw chat history.

The book's example state, unchanged:

In [1]:
state = {
    "task": "review refund dispute",
    "user_id": "U123",
    "facts": {
        "order_id": "O900",
        "refund_status": "pending",
    },
    "completed_steps": ["identity_verified", "policy_retrieved"],
    "open_questions": ["is exception approval allowed?"],
    "risk_level": "medium",
}
print(state)

{'task': 'review refund dispute', 'user_id': 'U123', 'facts': {'order_id': 'O900', 'refund_status': 'pending'}, 'completed_steps': ['identity_verified', 'policy_retrieved'], 'open_questions': ['is exception approval allowed?'], 'risk_level': 'medium'}


The book describes the planner/worker/reviewer split in prose without code -- here's a small, direct implementation of exactly that: each role reads and updates the *same* state object, and each step is recorded so there's one owner and no duplicate work.

In [2]:
def planner(state: dict) -> dict:
    """Breaks the request into sub-questions and assigns the first worker."""
    state = dict(state)
    state["open_questions"] = ["is exception approval allowed?"]
    state["completed_steps"] = state["completed_steps"] + ["planned"]
    return state


def worker_gather_policy(state: dict) -> dict:
    """A worker that owns exactly one step: retrieving policy for this dispute."""
    if "policy_retrieved" in state["completed_steps"]:
        return state  # already done -- do not repeat another worker's step
    state = dict(state)
    state["facts"] = {**state["facts"], "policy_ref": "refund_policy_v3#exceptions"}
    state["completed_steps"] = state["completed_steps"] + ["policy_retrieved"]
    return state


def reviewer(state: dict) -> dict:
    """Checks whether the case is actually ready to move forward."""
    state = dict(state)
    ready = "policy_retrieved" in state["completed_steps"] and not state["open_questions"]
    state["reviewer_verdict"] = "approved_for_next_step" if ready else "needs_more_evidence"
    return state

In [3]:
run_state = planner(state)
run_state = worker_gather_policy(run_state)  # already done in the book's example -- this is a no-op
run_state = reviewer(run_state)

print("completed_steps:", run_state["completed_steps"])
print("reviewer verdict:", run_state["reviewer_verdict"], "(still has an open question)")

completed_steps: ['identity_verified', 'policy_retrieved', 'planned']
reviewer verdict: needs_more_evidence (still has an open question)


**Handoff rules**: a general agent can start the conversation, but a specialist or a human should take over once the task needs deeper expertise or stricter control:

In [4]:
def decide_handoff(state: dict) -> str:
    if state["risk_level"] == "high":
        return "human_reviewer"
    if "policy_retrieved" not in state["completed_steps"]:
        return "policy_worker"
    if state["open_questions"]:
        return "specialist_agent"  # e.g. an exceptions specialist for the open question
    return "general_agent"


print("medium risk, open question ->", decide_handoff(run_state))

high_risk_state = {**run_state, "risk_level": "high"}
print("high risk ->", decide_handoff(high_risk_state))

medium risk, open question -> specialist_agent
high risk -> human_reviewer


The rule that keeps this useful: only add more agents when splitting roles adds real value. If handoffs create more confusion than control, one agent with a clear workflow is the better design.